# 03 - Train, validate and export the affinity model 📈

This notebook downloads the fixed embedding dataset, trains the regression head, reports held-out metrics, exports ONNX, runs a parity check, and uploads the final model.

In [ ]:
WK_DIR = "/content/"

In [ ]:
%cd {WK_DIR}
%cd Protein-Ligand-Affinity-Prediction-Using-LLM
!git clone https://github.com/karthikeyanr103/Protein-Ligand-Affinity-Prediction-Using-LLM.git
%cd {WK_DIR}/Protein-Ligand-Affinity-Prediction-Using-LLM
%pip install -e ".[runtime]" "huggingface-hub>=0.30"

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login, snapshot_download
from pathlib import Path
import os, subprocess, shutil

HF_USER = userdata.get('HF_USER')
DATASET_REPO = f'{HF_USER}/protein-compound-affinity-embeddings'
MODEL_REPO = f'{HF_USER}/protein-compound-affinity-onnx'
ROOT = Path(f'{WK_DIR}/drive/MyDrive/protein_affinity')
OUTPUT = ROOT / 'affinity_model'
OUTPUT.mkdir(parents=True, exist_ok=True)
token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = token
login(token=token)
DATASET = Path(snapshot_download(DATASET_REPO, repo_type='dataset', token=token))

## Training configuration

In [ ]:
config = ROOT / 'training.toml'
config.write_text(f'''[data]
path = "{DATASET}"
seed = 42

[model]
hidden_dims = [1024, 512, 128]
dropout = 0.25

[training]
batch_size = 512
epochs = 40
learning_rate = 0.0005
weight_decay = 0.0001
patience = 6
num_workers = 2

[output]
directory = "{OUTPUT}"
''')
print(config.read_text())

## Train and evaluate

In [ ]:
subprocess.run(['affinity-train', '--config', str(config)], check=True)
subprocess.run(['affinity-evaluate', '--artifacts', str(OUTPUT)], check=True)

In [ ]:
import json, pandas as pd
metadata = json.loads((OUTPUT / 'metadata.json').read_text())
history = pd.read_json(OUTPUT / 'history.json')
print('Test metrics:', metadata['test_metrics'])
history.tail()

## Export the trained head to ONNX

In [ ]:
subprocess.run(['affinity-export-onnx', '--artifacts', str(OUTPUT)], check=True)
shutil.copy2('MODEL_CARD.md', OUTPUT / 'README.md')
print(sorted(path.name for path in OUTPUT.iterdir()))

## Upload the trained model

In [ ]:
UPLOAD = False
if UPLOAD:
    api = HfApi(token=token)
    api.create_repo(MODEL_REPO, repo_type='model', exist_ok=True)
    api.upload_folder(
        repo_id=MODEL_REPO,
        repo_type='model',
        folder_path=str(OUTPUT),
        allow_patterns=['model.onnx', 'normalization.npz', 'metadata.json', 'README.md'],
    )
    print('Uploaded:', MODEL_REPO)